<h2 align="center" style='color:blue'> Monitoring and Data Drift Approach PSI, CSI Tutorial</h2>

------------------------------------

##### **Note: PSI/CSI computed using holdout test predictions to simulate production drift monitoring**


------------------------------------------

##### **PSI (Population Stabilty Index)**

**Purpose:** Measures the change in the distribution of data over different time periods.

**Focus:** Primarily on the dependent variable (e.g., model scores).

**Key Points:**

PSI helps to monitor if the model's performance remains stable over time.
A significant shift in PSI indicates potential model degradation or changes in the data distribution.

**Steps:**

1. Choose Reference Population:

    For model testing: Use the training data (In-sample scores) as the reference.
    For performance tracking: Use the Out-of-Time (OOT) scores as the reference.

2. Rank and Bin Reference Scores:

    Sort the reference scores (predicted probabilities).
    Divide them into 10 equal bins (deciles).

3. Determine Bin Cutoffs:

    Identify the cutoff points for each bin from the reference scores.

4. Calculate Reference Bin Percentages:

    Count the number of records in each bin.
    Calculate the percentage of records in each bin (should be around 10% each if bins are equal).

5. Bin the Test Scores:

    Apply the same cutoff points from the reference bins to the test scores.
    Assign test scores to the corresponding bins.

6. Calculate Test Bin Percentages:

    Count the number of records in each test bin.
    Calculate the percentage of records in each test bin.

7. Calculate PSI for Each Bin:

    Compute the difference in percentages between reference and test bins.
    Compute the natural log of the ratio of reference percentage to test percentage.
    Multiply the difference by the natural log value to get the PSI for each bin.

In [108]:
import pandas as pd
import numpy as np

In [109]:
train_probs_df = pd.read_csv('train_probs.csv') # get the predicted probabilities from our final model
test_probs_df = pd.read_csv('test_probs.csv') # get the predicted probabilities from our final model

In [110]:
train_probs = train_probs_df['0'].to_numpy()
test_probs = test_probs_df['0'].to_numpy()

In [111]:
# Define number of bins (e.g., 10 for deciles)
num_bins = 10 # Specify number of bins for dividing probabilities into deciles

np.linspace(0, 100, num_bins + 1)

array([  0.,  10.,  20.,  30.,  40.,  50.,  60.,  70.,  80.,  90., 100.])

In [112]:
bin_edges = np.percentile(train_probs, np.linspace(0, 100, num_bins + 1))
bin_edges

array([0.01185391, 0.16855122, 0.30560841, 0.41036099, 0.45581491,
       0.50279314, 0.5332595 , 0.58953237, 0.69103293, 0.83189063,
       0.99538506])

In [113]:
# Assign bins to train and test probabilities which was calculated above
train_bins = np.digitize(train_probs, bins=bin_edges, right=True) - 1
test_bins = np.digitize(test_probs, bins=bin_edges, right=True) - 1

In [114]:
train_bins

array([0, 3, 0, ..., 4, 8, 3], dtype=int64)

In [115]:
np.unique(train_bins)

array([-1,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9], dtype=int64)

In [116]:
# Ensure no negative bin indices
train_bins = np.clip(train_bins, 0, num_bins - 1)
test_bins = np.clip(test_bins, 0, num_bins - 1)
np.unique(train_bins)

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int64)

In [117]:
train_counts = pd.Series(train_bins).value_counts().sort_index()
test_counts = pd.Series(test_bins).value_counts().sort_index()
train_counts

0    11150
1    11150
2    11150
3    11150
4    11150
5    11181
6    11119
7    11150
8    11150
9    11150
Name: count, dtype: int64

In [118]:
train_percents = train_counts * 100 / len(train_probs)
test_percents = test_counts * 100 / len(test_probs)
train_percents

0    10.000000
1    10.000000
2    10.000000
3    10.000000
4    10.000000
5    10.027803
6     9.972197
7    10.000000
8    10.000000
9    10.000000
Name: count, dtype: float64

In [119]:
# Create dataframe with details
df = pd.DataFrame({
    'prob_range': [f'{bin_edges[i]:.6f} - {bin_edges[i+1]:.6f}' for i in range(len(bin_edges)-1)],
    'train_count': train_counts,
    'train_%': train_percents,
    'test_count': test_counts,
    'test_%': test_percents
})
df

,prob_range,train_count,train_%,test_count,test_%
0,0.011854 - 0.168551,11150,10.000000,3760,17.938931
1,0.168551 - 0.305608,11150,10.000000,3443,16.426527
2,0.305608 - 0.410361,11150,10.000000,2835,13.525763
3,0.410361 - 0.455815,11150,10.000000,2571,12.266221
4,0.455815 - 0.502793,11150,10.000000,2277,10.863550
5,0.502793 - 0.533260,11181,10.027803,2021,9.642176
6,0.533260 - 0.589532,11119,9.972197,1761,8.401718
7,0.589532 - 0.691033,11150,10.000000,1349,6.436069
8,0.691033 - 0.831891,11150,10.000000,759,3.621183
9,0.831891 - 0.995385,11150,10.000000,184,0.877863


In [120]:
# Calculate additional columns
df['A-B'] = df['train_%'] - df['test_%']
df['ln(A/B)'] = np.log(df['train_%'] / df['test_%'])
df['PSI'] = (df['A-B']/100) * df['ln(A/B)']

# Replace inf and NaN with 0 in PSI column
df['PSI'] = df['PSI'].replace([np.inf, -np.inf, np.nan], 0)
df

,prob_range,train_count,train_%,test_count,test_%,A-B,ln(A/B),PSI
0,0.011854 - 0.168551,11150,10.000000,3760,17.938931,-7.938931,-0.584388,0.046394
1,0.168551 - 0.305608,11150,10.000000,3443,16.426527,-6.426527,-0.496312,0.031896
2,0.305608 - 0.410361,11150,10.000000,2835,13.525763,-3.525763,-0.302011,0.010648
3,0.410361 - 0.455815,11150,10.000000,2571,12.266221,-2.266221,-0.204264,0.004629
4,0.455815 - 0.502793,11150,10.000000,2277,10.863550,-0.863550,-0.082828,0.000715
5,0.502793 - 0.533260,11181,10.027803,2021,9.642176,0.385627,0.039215,0.000151
6,0.533260 - 0.589532,11119,9.972197,1761,8.401718,1.570480,0.171365,0.002691
7,0.589532 - 0.691033,11150,10.000000,1349,6.436069,3.563931,0.440667,0.015705
8,0.691033 - 0.831891,11150,10.000000,759,3.621183,6.378817,1.015784,0.064795
9,0.831891 - 0.995385,11150,10.000000,184,0.877863,9.122137,2.432850,0.221928


In [121]:
# Calculate total PSI
total_psi = df['PSI'].sum()
total_psi

0.3995528770252007

### **CSI (Characteristic Stability Index)**

**Choose an Independent Variable:** Focus on one independent variable from your dataset for analysis.

**Set Up Bins:**

1. For Numeric Variables: Arrange the values and split them into 10 equal-sized bins.
2. For Categorical Variables: Use the current categories or merge them to form up to 10 groups.

**Identify Bin Cutoff Points:** Determine the threshold values that define the bins based on the reference (training) data.

**Compute Reference Distribution:** Count and calculate the percentage of records in each bin for the training data.

**Apply Cutoffs to Test Data:** Use the same bin thresholds from the training data to categorize the test data.

**Compute Test Distribution:** Count and calculate the percentage of records in each bin for the test data.

**Calculate CSI for Each Bin:**

1. Compute Differences: Find the difference between the percentages in each bin for training and test data.
2. Calculate Log Ratios: Compute the natural log of the ratio between the training and test percentages.
3. Determine Index: Multiply the difference by the log ratio to get the CSI for each bin.
4. Sum Up: Add the CSI values for all bins to get the overall Characteristic Stability Index.

In [74]:
X_train_1 = pd.read_csv('train.csv')
X_test = pd.read_csv('test.csv')

In [75]:
# lets understand for one feature

# Bin the features using quantile-based binning
train_binned = pd.qcut(X_train_1['Credit Score'], q=4, duplicates='drop')
test_binned = pd.qcut(X_test['Credit Score'], q=4, duplicates='drop')

In [76]:
train_binned.head()

0    (732.0, 1203.0]
1     (559.0, 732.0]
2     (559.0, 732.0]
3    (732.0, 1203.0]
4     (559.0, 732.0]
Name: Credit Score, dtype: category
Categories (4, interval[float64, right]): [(-94.001, 459.0] < (459.0, 559.0] < (559.0, 732.0] < (732.0, 1203.0]]

In [77]:
test_binned.head()

0    (735.0, 1178.0]
1    (735.0, 1178.0]
2    (735.0, 1178.0]
3     (559.0, 735.0]
4    (459.75, 559.0]
Name: Credit Score, dtype: category
Categories (4, interval[float64, right]): [(-64.001, 459.75] < (459.75, 559.0] < (559.0, 735.0] < (735.0, 1178.0]]

In [78]:
# Calculate proportions for each bin
train_proportions = train_binned.value_counts(normalize=True, sort=False)
test_proportions = test_binned.value_counts(normalize=True, sort=False)

In [79]:
train_proportions.head()


Credit Score
(-94.001, 459.0]    0.252162
(459.0, 559.0]      0.252655
(559.0, 732.0]      0.245851
(732.0, 1203.0]     0.249332
Name: proportion, dtype: float64

In [80]:
test_proportions.head()

Credit Score
(-64.001, 459.75]    0.250000
(459.75, 559.0]      0.252624
(559.0, 735.0]       0.248044
(735.0, 1178.0]      0.249332
Name: proportion, dtype: float64

In [81]:
# Ensure all bins are represented
all_bins = train_proportions.index.union(test_proportions.index)
train_proportions = train_proportions.reindex(all_bins, fill_value=0)
test_proportions = test_proportions.reindex(all_bins, fill_value=0)

# Reason why we do above step (good to adopt in day-to-day task to consider all the corner cases thinking before hand)
# Consistency in Comparison:
# To calculate the Characteristic Stability Index (CSI), you need to compare the proportions of each bin or category between the training and test datasets.
# If a bin or category is missing in one of the datasets, it could cause issues in the comparison, leading to inaccurate CSI calculations.

# Handling Edge Cases:
# In real-world data, it's possible that some bins or categories exist in the training dataset but not in the test dataset, or vice versa.
# This code ensures that these edge cases are handled gracefully by assigning a proportion of 0 to missing bins or categories.

# Avoiding Errors in CSI Calculation:
# The CSI calculation involves dividing the proportions (train_pct / test_pct). If one of the proportions is missing, it could lead to division by zero

In [82]:
# CSI formula is same as PSI

for bin in all_bins:
    train_pct = train_proportions[bin]
    display(train_pct)

0.2521617497456765

0.252654501525941

0.2458513479145473

0.2493324008138352

0.0

0.0

0.0

0.0

In [83]:
for bin in all_bins:
  train_pct = train_proportions[bin]
  test_pct = test_proportions[bin]
  display(test_pct)

0.0

0.0

0.0

0.0

0.25

0.2526240458015267

0.248043893129771

0.2493320610687023

In [84]:
for bin in all_bins:
  train_pct = train_proportions[bin]
  test_pct = test_proportions[bin]
  A_B = train_pct - test_pct # A minus B
  display(A_B)

0.2521617497456765

0.252654501525941

0.2458513479145473

0.2493324008138352

-0.25

-0.2526240458015267

-0.248043893129771

-0.2493320610687023

In [85]:
for bin in all_bins:
    train_pct = train_proportions[bin]
    test_pct = test_proportions[bin]
    A_B = train_pct - test_pct
    ln_A_B = np.log(train_pct / test_pct) if test_pct > 0 and train_pct > 0 else 0
    display(ln_A_B)

0

0

0

0

0

0

0

0

In [86]:
for bin in all_bins:
    train_pct = train_proportions[bin]
    test_pct = test_proportions[bin]
    A_B = train_pct - test_pct
    ln_A_B = np.log(train_pct / test_pct) if test_pct > 0 and train_pct > 0 else 0
    csi = A_B * ln_A_B
    display(csi) # below csi values are for each bin of age feature

0.0

0.0

0.0

0.0

-0.0

-0.0

-0.0

-0.0

In [99]:
numerical_features =  ["Derogatory","Credit Score","Late_Payment_30DPD_Last_12M","Credit Card Payment Failures","Recent Payment Irregularity ",
"Late_Payment_30DPD_Last_24M","Long_Term_Payment_Delinquency_Count","Active Credit Cards","Total Historical Defaults","Open Defaults"]
categorical_features = ["Residential Status","Occupation Type","Document Type","Employment Status","Applicant Age","Bureau Default ","Score Card","Bureau Enquiries (Last 12 Months)"]        

In [100]:
# Initialize lists to store CSI results
numerical_results = []
categorical_results = []

In [101]:
print(X_train_1.columns.tolist())


['Residential Status', 'Occupation Type', 'Document Type', 'Employment Status', 'Applicant Age', 'Bureau Default ', 'Derogatory', 'Score Card', 'Credit Score', 'Late_Payment_30DPD_Last_12M', 'Credit Card Payment Failures', 'Recent Payment Irregularity ', 'Late_Payment_30DPD_Last_24M', 'Long_Term_Payment_Delinquency_Count', 'Bureau Enquiries (Last 12 Months)', 'Active Credit Cards', 'Total Historical Defaults', 'Open Defaults']


In [102]:
# Calculate CSI for numerical features
for feature in numerical_features:
    # Bin the features using quantile-based binning
    train_binned = pd.qcut(X_train_1[feature], q=4, duplicates='drop')
    test_binned = pd.qcut(X_test[feature], q=4, duplicates='drop')

    # Calculate proportions for each bin
    train_proportions = train_binned.value_counts(normalize=True, sort=False)
    test_proportions = test_binned.value_counts(normalize=True, sort=False)

    # Ensure all bins are represented
    all_bins = train_proportions.index.union(test_proportions.index)
    train_proportions = train_proportions.reindex(all_bins, fill_value=0)
    test_proportions = test_proportions.reindex(all_bins, fill_value=0)

    # Calculate CSI for each bin
    for bin in all_bins:
        train_pct = train_proportions[bin]
        test_pct = test_proportions[bin]
        A_B = train_pct - test_pct
        ln_A_B = np.log(train_pct / test_pct) if test_pct > 0 and train_pct > 0 else 0
        csi = A_B * ln_A_B

        numerical_results.append({
            'Feature': feature,
            'Category/ Bin': bin,
            'Train Count': train_binned[train_binned == bin].count(),
            'Train %': train_pct * 100,
            'Test Count': test_binned[test_binned == bin].count(),
            'Test %': test_pct * 100,
            'A-B': A_B,
            'ln(A/B)': ln_A_B,
            'CSI': csi
        })

In [103]:
# Calculate CSI for categorical features
for feature in categorical_features:
    # Calculate proportions for each category
    train_proportions = X_train_1[feature].value_counts(normalize=True)
    test_proportions = X_test[feature].value_counts(normalize=True)

    # Ensure all categories are represented
    all_categories = train_proportions.index.union(test_proportions.index)
    train_proportions = train_proportions.reindex(all_categories, fill_value=0)
    test_proportions = test_proportions.reindex(all_categories, fill_value=0)

    # Calculate CSI for each category
    for category in all_categories:
        train_pct = train_proportions[category]
        test_pct = test_proportions[category]
        A_B = train_pct - test_pct
        ln_A_B = np.log(train_pct / test_pct) if test_pct > 0 and train_pct > 0 else 0
        csi = A_B * ln_A_B

        categorical_results.append({
            'Feature': feature,
            'Category/ Bin': category,
            'Train Count': X_train_1[feature].value_counts().get(category, 0),
            'Train %': train_proportions[category] * 100,
            'Test Count': X_test[feature].value_counts().get(category, 0),
            'Test %': test_proportions[category] * 100,
            'A-B': A_B,
            'ln(A/B)': ln_A_B,
            'CSI': csi
        })

In [104]:
 # Combine results into a single DataFrame
numerical_df = pd.DataFrame(numerical_results)
categorical_df = pd.DataFrame(categorical_results)
csi_combined_df = pd.concat([numerical_df, categorical_df], ignore_index=True)

In [105]:
# Display combined CSI results
print("Combined CSI Table:")
csi_combined_df['CSI'] = csi_combined_df['CSI'].round(6)
csi_combined_df

Combined CSI Table:


,Feature,Category/ Bin,Train Count,Train %,Test Count,Test %,A-B,ln(A/B),CSI
0,Derogatory,"(-0.001, 31.0]",62912,100.000000,0,0.000000,1.000000,0.000000,0.000000
1,Derogatory,"(-0.001, 253.0]",0,0.000000,20960,100.000000,-1.000000,0.000000,-0.000000
2,Credit Score,"(-94.001, 459.0]",15864,25.216175,0,0.000000,0.252162,0.000000,0.000000
3,Credit Score,"(459.0, 559.0]",15895,25.265450,0,0.000000,0.252655,0.000000,0.000000
4,Credit Score,"(559.0, 732.0]",15467,24.585135,0,0.000000,0.245851,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...
72,Bureau Enquiries (Last 12 Months),5-Apr,7880,12.525432,2565,12.237595,0.002878,0.023248,0.000067
73,Bureau Enquiries (Last 12 Months),7-Jun,2565,4.077124,804,3.835878,0.002412,0.060993,0.000147
74,Bureau Enquiries (Last 12 Months),11-Aug,1379,2.191951,465,2.218511,-0.000266,-0.012045,0.000003
75,Bureau Enquiries (Last 12 Months),12+,227,0.360821,82,0.391221,-0.000304,-0.080890,0.000025


In [106]:
# Group by 'Feature' and aggregate by summing up relevant columns
csi_summary_df = csi_combined_df.groupby('Feature').agg({
    'Train Count': 'sum',
    'Test Count': 'sum',
    'CSI': 'sum'
}).reset_index()

In [107]:
csi_summary_df

,Feature,Train Count,Test Count,CSI
0,Active Credit Cards,62912,20960,0.000017
1,Applicant Age,62912,20960,0.000134
2,Bureau Default,62912,20960,0.000080
3,Bureau Enquiries (Last 12 Months),62912,20960,0.000357
4,Credit Card Payment Failures,62912,20960,0.000051
5,Credit Score,62912,20960,0.000000
6,Derogatory,62912,20960,0.000000
7,Document Type,62912,20960,0.000097
8,Employment Status,62912,20960,0.001132
9,Late_Payment_30DPD_Last_12M,62912,20960,0.000000
